# Notebook 02 — Baselines + StatsForecast Models

**ISA 444 Final Project — Retail Forecasting (Walmart)**

Mirrors the structure of `duke_energy_statforecast.ipynb` from class.

### Models fitted in this notebook
| Model | Type | Uses exogenous features? |
|---|---|---|
| `Naive` | Baseline | No |
| `SeasonalNaive` (season_length=52) | Baseline | No |
| `AutoETS` (season_length=52) | Statistical | No (ETS doesn't support exog) |
| `AutoARIMA` (season_length=52) | Statistical | No — see note below |
| `AutoARIMA_X` (with exog) | Statistical | Yes — `is_holiday_week`, markdowns, temperature |

We run `AutoARIMA` twice on purpose: once **without** exogenous features (apples-to-apples vs ETS / SNaive) and once **with** the holiday / markdown / temperature regressors so we can see how much exog actually helps. This is the same trick the Duke Energy notebook used (`AutoARIMA` vs `AutoARIMAwPred`).

### Cross-validation setup
- **horizon `h = 4`** weeks (monthly forecasts as requested)
- **`n_windows = 5`** → 5-fold CV
- **`step_size = 4`** → non-overlapping folds
- **`freq = 'W-FRI'`** → Walmart weeks end on Friday
- **`season_length = 52`** → annual retail cycle

### Outputs saved
- `..\outputs\cv_results\cv_statsforecast_predictions.csv` — every fold's per-week prediction (long format, columns: unique_id, ds, cutoff, y, Naive, SeasonalNaive, AutoETS, AutoARIMA, AutoARIMA_X)
- `..\outputs\cv_results\eval_statsforecast.csv` — per-series, per-metric evaluation (ME / MAE / RMSE / MAPE)
- `..\outputs\forecasts\test_statsforecast.csv` — future forecasts on the Kaggle test dates


## Install Packages

Run once on a fresh environment, then comment out.

In [1]:
# !pip install statsforecast utilsforecast pyarrow

## Library Imports

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, AutoETS, AutoARIMA

from utilsforecast.evaluation import evaluate
from utilsforecast.losses import bias, mae, rmse, mape

## Paths and Settings

In [3]:
DATA_DIR = Path("../data")
OUT_DIR  = Path("../outputs")
(OUT_DIR / "cv_results").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "forecasts").mkdir(parents=True, exist_ok=True)

# Forecasting settings — kept consistent across ALL model notebooks.
FREQ           = "W-FRI"   # Walmart weeks end on Friday
H              = 4         # 4-week (monthly) forecast horizon
N_WINDOWS      = 5         # 5-fold CV per rubric
STEP_SIZE      = 4         # non-overlapping folds (step_size = h)
SEASON_LENGTH  = 52        # weekly data, annual seasonality

## Load the Prepared Training Data

Built by Notebook 01. Top-20 Store/Dept series with covariates and calendar features. Univariate models (Naive, SNaive, AutoETS, AutoARIMA-no-exog) just need `unique_id, ds, y` so we'll select that subset when calling them.

In [4]:
df_train = pd.read_parquet(DATA_DIR / "walmart_top20_train.parquet")
df_test  = pd.read_parquet(DATA_DIR / "walmart_top20_test.parquet")

print("Training series :", df_train["unique_id"].nunique())
print("Training rows   :", len(df_train))
print("Date range      :", df_train["ds"].min().date(), "->", df_train["ds"].max().date())
print()
df_train.head()

Training series : 20
Training rows   : 2860
Date range      : 2010-02-05 -> 2012-10-26



,unique_id,ds,y,is_holiday_week,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,had_markdown,Temperature,Fuel_Price,CPI,Unemployment,store_type,store_size,week_of_year,month,quarter,year
0,S10_D72,2010-02-05,232558.51,0,0.0,0.0,0.0,0.0,0.0,0,54.34,2.962,126.442065,9.765,B,126512,5,2,1,2010
1,S10_D72,2010-02-12,202622.42,1,0.0,0.0,0.0,0.0,0.0,0,49.96,2.828,126.496258,9.765,B,126512,6,2,1,2010
2,S10_D72,2010-02-19,184982.01,0,0.0,0.0,0.0,0.0,0.0,0,58.22,2.915,126.526286,9.765,B,126512,7,2,1,2010
3,S10_D72,2010-02-26,186002.43,0,0.0,0.0,0.0,0.0,0.0,0,52.77,2.825,126.552286,9.765,B,126512,8,2,1,2010
4,S10_D72,2010-03-05,191989.54,0,0.0,0.0,0.0,0.0,0.0,0,55.92,2.877,126.578286,9.765,B,126512,9,3,1,2010


## Is MAPE Appropriate For Every Series?

MAPE = mean of `|y - y_hat| / |y|`. It blows up when `y` gets near zero and is generally untrustworthy below `y ≈ 100`-ish. Since we picked the **top 20 by total sales**, all series should be well above that floor — but we check explicitly rather than assume.


In [5]:
mape_check = (
    df_train.groupby("unique_id")["y"]
    .agg(min_y="min", median_y="median")
    .sort_values("min_y")
)
# Flag any series whose minimum weekly sales is too low for MAPE to behave.
mape_check["mape_ok"] = mape_check["min_y"] > 1000
display(mape_check)

if mape_check["mape_ok"].all():
    print("\n[OK] All 20 series have min weekly sales > $1,000 — MAPE is well-behaved everywhere.")
else:
    print("\n[WARN] Some series have very small minimum sales — MAPE may be unstable for these. Lean on MAE/RMSE.")

,min_y,median_y,mape_ok
unique_id,,,
S41_D92,86342.13,114286.18,True
S19_D92,87867.72,112325.25,True
S27_D95,89665.22,116561.94,True
S10_D72,91508.44,123509.08,True
S1_D95,93358.91,120721.07,True
S13_D90,94952.54,114173.15,True
S24_D92,96455.09,119716.20,True
S31_D92,100890.32,125360.15,True
S13_D95,104516.06,135555.41,True



[OK] All 20 series have min weekly sales > $1,000 — MAPE is well-behaved everywhere.


## Define the Models

We separate the models into two `StatsForecast` instances:

1. **`sf_no_exog`** — baselines + AutoETS + AutoARIMA (univariate). Same recipe as the Duke notebook.
2. **`sf_with_exog`** — AutoARIMA with exogenous regressors (only ARIMA supports exog among the stats models).

The exogenous features we pass are all values that are **known in advance for the forecast horizon**:
- `is_holiday_week` — Kaggle gives us future holiday flags
- `MarkDown1..5` — markdown promo amounts (Kaggle includes these for test dates too)
- `Temperature` — Kaggle includes future temperatures

We deliberately exclude `CPI`, `Unemployment`, and `Fuel_Price`: they move too slowly to add value at a 4-week horizon and we forward-filled some of their test values, so giving them weight could be misleading.


In [6]:
# Univariate stats models.
models_no_exog = [
    Naive(alias="Naive"),
    SeasonalNaive(season_length=SEASON_LENGTH, alias="SeasonalNaive"),
    AutoETS(season_length=SEASON_LENGTH, alias="AutoETS"),
    AutoARIMA(season_length=SEASON_LENGTH, alias="AutoARIMA"),
]

# AutoARIMA again, but this time we'll feed in exogenous columns at fit/predict time.
# n_jobs=1 keeps memory predictable on an 8 GB laptop; raise to -1 if you have headroom.
models_with_exog = [
    AutoARIMA(season_length=SEASON_LENGTH, alias="AutoARIMA_X"),
]

sf_no_exog   = StatsForecast(models=models_no_exog,   freq=FREQ, n_jobs=1)
sf_with_exog = StatsForecast(models=models_with_exog, freq=FREQ, n_jobs=1)

# Columns we'll feed into the exog-aware run.
EXOG_COLS = ["is_holiday_week", "MarkDown1", "MarkDown2", "MarkDown3",
             "MarkDown4", "MarkDown5", "Temperature"]

## Cross-Validation — Univariate Models

`StatsForecast.cross_validation` with `n_windows=5` and `step_size=4` produces 5 non-overlapping 4-week holdout folds, walking back from the end of each series. Same approach as the Duke notebook.


In [7]:
%%time
cv_no_exog = sf_no_exog.cross_validation(
    df          = df_train[["unique_id", "ds", "y"]],
    h           = H,
    n_windows   = N_WINDOWS,
    step_size   = STEP_SIZE,
)
print("CV (no exog) shape:", cv_no_exog.shape)
cv_no_exog.head()

CV (no exog) shape: (400, 8)
CPU times: total: 4min 18s
Wall time: 4min 44s


,unique_id,ds,cutoff,y,Naive,SeasonalNaive,AutoETS,AutoARIMA
0,S10_D72,2012-06-15,2012-06-08,105499.39,125434.23,127450.66,125453.077037,135381.194651
1,S10_D72,2012-06-22,2012-06-08,107949.41,125434.23,117948.54,114657.507531,117357.420039
2,S10_D72,2012-06-29,2012-06-08,96579.10,125434.23,114398.47,116075.626160,109978.895935
3,S10_D72,2012-07-06,2012-06-08,100464.25,125434.23,108519.93,109074.835739,102380.377658
4,S10_D72,2012-07-13,2012-07-06,92923.05,100464.25,115004.83,110645.290441,107966.684059


## Cross-Validation — AutoARIMA With Exogenous Features

Same CV settings; the extra columns are passed via the `df` argument and StatsForecast figures out which are exogenous (anything beyond `unique_id, ds, y`).


In [8]:
%%time
cv_with_exog = sf_with_exog.cross_validation(
    df          = df_train[["unique_id", "ds", "y"] + EXOG_COLS],
    h           = H,
    n_windows   = N_WINDOWS,
    step_size   = STEP_SIZE,
)
print("CV (with exog) shape:", cv_with_exog.shape)
cv_with_exog.head()

CV (with exog) shape: (400, 5)
CPU times: total: 17min 1s
Wall time: 18min 24s


,unique_id,ds,cutoff,y,AutoARIMA_X
0,S10_D72,2012-06-15,2012-06-08,105499.39,140731.179577
1,S10_D72,2012-06-22,2012-06-08,107949.41,115602.221160
2,S10_D72,2012-06-29,2012-06-08,96579.10,113809.226981
3,S10_D72,2012-07-06,2012-06-08,100464.25,94962.454535
4,S10_D72,2012-07-13,2012-07-06,92923.05,121319.979562


## Combine The Two CV Tables Into One

Merging on `(unique_id, ds, cutoff, y)` lines up rows from both runs into a single tidy table that the evaluation step can consume directly.


In [9]:
cv_stats = (
    cv_no_exog
    .merge(cv_with_exog, on=["unique_id", "ds", "cutoff", "y"], how="left")
)

# Save the full CV prediction set for downstream plotting and inspection.
cv_stats.to_csv(OUT_DIR / "cv_results" / "cv_statsforecast_predictions.csv", index=False)
print("Saved CV predictions:", (OUT_DIR / "cv_results" / "cv_statsforecast_predictions.csv").resolve())
print("Shape:", cv_stats.shape)
cv_stats.head()

Saved CV predictions: C:\Users\23mwa\outputs\cv_results\cv_statsforecast_predictions.csv
Shape: (400, 9)


,unique_id,ds,cutoff,y,Naive,SeasonalNaive,AutoETS,AutoARIMA,AutoARIMA_X
0,S10_D72,2012-06-15,2012-06-08,105499.39,125434.23,127450.66,125453.077037,135381.194651,140731.179577
1,S10_D72,2012-06-22,2012-06-08,107949.41,125434.23,117948.54,114657.507531,117357.420039,115602.221160
2,S10_D72,2012-06-29,2012-06-08,96579.10,125434.23,114398.47,116075.626160,109978.895935,113809.226981
3,S10_D72,2012-07-06,2012-06-08,100464.25,125434.23,108519.93,109074.835739,102380.377658,94962.454535
4,S10_D72,2012-07-13,2012-07-06,92923.05,100464.25,115004.83,110645.290441,107966.684059,121319.979562


## Evaluate Per Series, Per Metric

`utilsforecast.evaluation.evaluate` takes a CV-prediction table and returns one row per (series, metric) with one column per model. We use:

- **`bias`** = mean error (negative => model under-forecasts on average)
- **`mae`** = mean absolute error
- **`rmse`** = root mean squared error
- **`mape`** = mean absolute percentage error (we already confirmed it's safe for our top-20 series)

This is the same evaluation call used in the Duke notebook.


In [10]:
model_cols = ["Naive", "SeasonalNaive", "AutoETS", "AutoARIMA", "AutoARIMA_X"]

eval_stats = evaluate(
    cv_stats,
    metrics = [bias, mae, rmse, mape],
    models  = model_cols,
)

# Save the per-series, per-metric evaluation table.
eval_stats.to_csv(OUT_DIR / "cv_results" / "eval_statsforecast.csv", index=False)
print("Saved evaluation table:", (OUT_DIR / "cv_results" / "eval_statsforecast.csv").resolve())
print("Shape:", eval_stats.shape)
eval_stats.head(8)

Saved evaluation table: C:\Users\23mwa\outputs\cv_results\eval_statsforecast.csv
Shape: (400, 8)


,unique_id,cutoff,metric,Naive,SeasonalNaive,AutoETS,AutoARIMA,AutoARIMA_X
0,S10_D72,2012-06-08,bias,22811.1925,14456.3625,13692.224117,13651.434571,13653.233063
1,S13_D90,2012-06-08,bias,3241.9225,-4937.3225,1238.828473,2994.518564,3094.057343
2,S13_D92,2012-06-08,bias,8317.1675,-17660.3000,-736.667669,2228.207282,1322.042726
3,S13_D95,2012-06-08,bias,-4235.6975,-14413.6200,-8871.727004,6869.226943,1185.882918
4,S14_D92,2012-06-08,bias,57192.4700,26923.4975,35771.222723,30839.971569,26369.783236
5,S14_D95,2012-06-08,bias,23683.4575,33496.6750,14896.253811,23686.416987,20581.317745
6,S19_D92,2012-06-08,bias,11507.3850,-577.6400,-682.256064,1909.043318,3966.593414
7,S1_D92,2012-06-08,bias,13123.6350,-15594.1875,-5111.004878,-3943.781270,-1547.502009


## Aggregate View Across All Series

In [11]:
# Mean of each metric across all 20 series — quick at-a-glance comparison.
agg_stats = (
    eval_stats
    .drop(columns=["unique_id"])
    .groupby("metric")
    .mean()
    .round(2)
)
display(agg_stats)
agg_stats.to_csv(OUT_DIR / "cv_results" / "eval_statsforecast_aggregate.csv")

,cutoff,Naive,SeasonalNaive,AutoETS,AutoARIMA,AutoARIMA_X
metric,,,,,,
bias,2012-08-03,2025.84,-1012.75,857.83,2227.37,2355.21
mae,2012-08-03,12065.41,11298.37,7845.23,7409.31,7597.05
mape,2012-08-03,0.09,0.08,0.06,0.05,0.06
rmse,2012-08-03,13622.43,12641.13,9117.87,8859.14,9078.67


## Count Wins Per Metric

For each (series, metric), the model with the **lowest** value wins. For `bias` we want the value closest to zero, so we compare absolute bias.


In [12]:
def count_wins(eval_df: pd.DataFrame, model_cols: list) -> pd.DataFrame:
    """Count how often each model 'wins' (= achieves best metric value per series).

    Bias is closest-to-zero (use absolute value).
    All other metrics are minimum.
    """
    wins = {}
    for metric_name, group in eval_df.groupby("metric"):
        if metric_name == "bias":
            # Closest to zero wins -> compare absolute values.
            best = group[model_cols].abs().idxmin(axis=1)
        else:
            best = group[model_cols].idxmin(axis=1)
        wins[metric_name] = best.value_counts()
    return pd.DataFrame(wins).fillna(0).astype(int)

wins_stats = count_wins(eval_stats, model_cols)
display(wins_stats)
wins_stats.to_csv(OUT_DIR / "cv_results" / "wins_statsforecast.csv")

,bias,mae,mape,rmse
AutoARIMA,28,33,32,25
AutoARIMA_X,17,25,25,22
AutoETS,26,18,18,24
Naive,12,10,11,11
SeasonalNaive,17,14,14,18


## Future Forecasts For The Kaggle Test Dates

Now we fit each model on the **full training history** and predict the next 39 weeks (the Kaggle test horizon). These forecasts go into the final deliverable CSV.

Note: there are no actuals here, so this section produces predictions only — no metrics.


In [13]:
%%time
# Univariate fit.
sf_no_exog.fit(df=df_train[["unique_id", "ds", "y"]])

# How many weeks ahead does the Kaggle test set extend?
h_future = df_test.groupby("unique_id").size().max()
print("Future horizon (weeks):", h_future)

fc_no_exog = sf_no_exog.predict(h=h_future)
fc_no_exog.head()

Future horizon (weeks): 39
CPU times: total: 46.3 s
Wall time: 52.2 s


,unique_id,ds,Naive,SeasonalNaive,AutoETS,AutoARIMA
0,S10_D72,2012-11-02,121126.83,164085.50,121731.372285,146825.525015
1,S10_D72,2012-11-09,121126.83,165484.28,154154.779805,152524.983072
2,S10_D72,2012-11-16,121126.83,142730.01,125783.764293,131585.249382
3,S10_D72,2012-11-23,121126.83,630999.19,721370.335147,620620.016105
4,S10_D72,2012-11-30,121126.83,156039.04,189763.215386,145982.881427


In [14]:
%%time
# Exog-aware fit — needs the same exog columns at predict time, supplied via X_df.
sf_with_exog.fit(df=df_train[["unique_id", "ds", "y"] + EXOG_COLS])

future_X = df_test[["unique_id", "ds"] + EXOG_COLS]

fc_with_exog = sf_with_exog.predict(h=h_future, X_df=future_X)
fc_with_exog.head()

CPU times: total: 3min 24s
Wall time: 3min 38s


,unique_id,ds,AutoARIMA_X
0,S10_D72,2012-11-02,131098.849783
1,S10_D72,2012-11-09,126619.559728
2,S10_D72,2012-11-16,155286.438595
3,S10_D72,2012-11-23,605614.494304
4,S10_D72,2012-11-30,168659.878861


In [15]:
# Merge the two forecast tables and save.
test_fc_stats = fc_no_exog.merge(fc_with_exog, on=["unique_id", "ds"], how="left")

test_fc_stats.to_csv(OUT_DIR / "forecasts" / "test_statsforecast.csv", index=False)
print("Saved test forecasts:", (OUT_DIR / "forecasts" / "test_statsforecast.csv").resolve())
print("Shape:", test_fc_stats.shape)
test_fc_stats.head()

Saved test forecasts: C:\Users\23mwa\outputs\forecasts\test_statsforecast.csv
Shape: (780, 7)


,unique_id,ds,Naive,SeasonalNaive,AutoETS,AutoARIMA,AutoARIMA_X
0,S10_D72,2012-11-02,121126.83,164085.50,121731.372285,146825.525015,131098.849783
1,S10_D72,2012-11-09,121126.83,165484.28,154154.779805,152524.983072,126619.559728
2,S10_D72,2012-11-16,121126.83,142730.01,125783.764293,131585.249382,155286.438595
3,S10_D72,2012-11-23,121126.83,630999.19,721370.335147,620620.016105,605614.494304
4,S10_D72,2012-11-30,121126.83,156039.04,189763.215386,145982.881427,168659.878861


## Notebook 02 — Done

**Outputs:**
- `..\outputs\cv_results\cv_statsforecast_predictions.csv` — every CV fold prediction
- `..\outputs\cv_results\eval_statsforecast.csv` — per-series, per-metric scores
- `..\outputs\cv_results\eval_statsforecast_aggregate.csv` — mean across series
- `..\outputs\cv_results\wins_statsforecast.csv` — win counts per model per metric
- `..\outputs\forecasts\test_statsforecast.csv` — future forecasts on Kaggle test dates

**Next:** Notebook 03 will run LightGBM via MLForecast with lag features and the same exogenous columns.
